In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
from flax import nnx

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
test_file_path = "/Users/jozbee/work/eng/comp/data/clean_00_sms_drive.hdf"
acc_ref, omega_ref = load_clean_references(file_path)
test_acc_ref, test_omega_ref = load_clean_references(test_file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)
test_acc_ref = jnp.clip(test_acc_ref, -1.0, 1.0)

In [ ]:
def smooth_data(data, nu=0):
    ts = np.arange(data.size) * dt
    return sci_interp.make_smoothing_spline(ts, data, lam=1e0)(ts, nu=nu)

In [ ]:
# data_range = [0, 90 * 200]
# data_range = [0, 4000]
# data_range = [8000, 12000]
# data_range = [8000, 90 * 200]
# data_range = [8000, 150 * 200]
data_range = [0, 250 * 200]

dt = 0.005
ts = np.arange(*data_range) * dt
data = acc_ref[data_range[0]: data_range[1], 0]
# data = test_acc_ref[data_range[0]: data_range[1], 0]
ref_data = smooth_data(data)
ref_datap = smooth_data(data, nu=1)

## conv

In [ ]:
min_trip = 5.0
max_trip = 10.0

class ConvNet(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.lay0 = nnx.Conv(in_features=1, out_features=1, kernel_size=11, strides=1, padding="VALID", rngs=rngs)
        self.lay10 = functools.partial(nnx.max_pool, window_shape=(11,), strides=(6,))
        self.lay11 = functools.partial(nnx.min_pool, window_shape=(11,), strides=(6,))
        self.lay2 = nnx.Linear(in_features=10, out_features=7, rngs=rngs)
        self.lay3 = nnx.Linear(in_features=7, out_features=3, rngs=rngs)
        self.lay4 = nnx.Linear(in_features=3, out_features=1, rngs=rngs)

    def __call__(self, x: jax.Array):
        assert x.shape == (50,)
        x = x.reshape(1, -1, 1)
        x = self.lay0(x)
        x0 = self.lay10(x)
        x1 = self.lay11(x)
        x = jnp.concatenate([jnp.squeeze(x0), jnp.squeeze(x1)])
        x = jnp.tanh(self.lay2(x))
        x = self.lay4(jnp.tanh(self.lay3(x)))
        x = (jnp.tanh(x * 1e1) + 1.0) * (max_trip - min_trip) / 2.0 + min_trip
        x *= -1  # sign convention for roots
        return jnp.squeeze(x)

rngs = nnx.Rngs(42)
conv = ConvNet(rngs=rngs)
params0, build_conv_net = jax.flatten_util.ravel_pytree(conv)

## filt

In [ ]:
def fast_trip_E0(f: jax.Array) -> jax.Array:
    x0 = f**2
    x1 = x0 + 80000
    x2 = jnp.exp((1/200)*f)
    x3 = (1/80000)*x2
    x4 = (1/40000)*x2
    x5 = f**3
    x6 = x3*(f + 400)
    return jnp.array([[x3*(800*f + x1), x0*x4*(-f - 600), x5*x6], [x6, x4*(-200*f - x0 + 40000), x3*x5], [x3, x4*(200 - f), x3*(-400*f + x1)]])

def fast_trip_E1(f: jax.Array) -> jax.Array:
    x0 = (1/200)*f
    x1 = jnp.exp(x0)
    x2 = (1/80000)*x1
    return jnp.array([[x2*(f + 400)], [x2], [(f**2*x2 - x0*x1 + x1 - 1)/f**3]])

def fast_trip_C(f: jax.Array, nu: int) -> jax.Array:
    assert 0 <= nu and nu <= 2
    if nu == 0:
        return jnp.array([0, 0, (-f)**3])
    elif nu == 1:
        return jnp.array([0, (-f)**3, 0])
    else:  # nu == 2
        return jnp.array([(-f)**3, 0, 0])

@functools.partial(jax.jit, static_argnames=["nu"])
def fast_trip_E0_E1_C(f: jax.Array, nu: int=0) -> tuple[jax.Array, jax.Array, jax.Array]:
    return fast_trip_E0(f), jnp.ravel(fast_trip_E1(f)), fast_trip_C(f, nu)

@jax.jit
def fast_obs_x0(f, y_0, y_1, y_2, u_0, u_1):
    x0 = f**3
    x1 = x0**(-1.0)
    x2 = f*y_2
    x3 = f**2
    x4 = jnp.exp((1/200)*f)
    x5 = jnp.exp((1/100)*f)
    x6 = x5*y_0
    x7 = 160000*x4
    x8 = f*u_0
    x9 = x3*x4
    return jnp.ravel(jnp.array([[(1/400)*x1*(80000*f*u_0*x5 - f*u_1*x7 + 240000*f*u_1 + 320000*f*x4*y_1 - 80000*f*x6 - u_0*x0*x4 - 16000000*u_0*x4 + 16000000*u_0*x5 - 600*u_0*x9 + u_1*x0*x4 + 600*u_1*x3*x4 + 400*u_1*x3 - 16000000*u_1*x4 + 16000000*u_1 - 240000*x2 - 400*x3*y_2 + 32000000*x4*y_1 - 16000000*x6 - x7*x8 - 16000000*y_2)], [x1*((1/2)*f*u_1*x4 + f*u_1 - 100*u_0*x4 + 100*u_0*x5 - 1/800*u_0*x9 + (1/800)*u_1*x3*x4 - 300*u_1*x4 + 300*u_1 - x2 - 1/2*x4*x8 + 400*x4*y_1 - 100*x6 - 300*y_2)], [-x1*y_2]]))


In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def nn_linear_filt(
    nn: ConvNet,
    data: jax.Array,
    nu: int=0,
) -> jax.Array:
    y = jnp.zeros(data.size + 50)
    up = jnp.concatenate([jnp.ones(50) * data[0], data])  # data (u) padded

    def filt_body(i: int, y: jax.Array) -> jax.Array:
        u_hist = jax.lax.dynamic_slice(up, [i], [50])
        y_hist = jax.lax.dynamic_slice(y, [i], [50])
        f = nn(u_hist - y_hist)
        E0, E1, C = fast_trip_E0_E1_C(f, nu)

        idx = i + 50  # index past history
        u = up[idx]  # not included in `u_hist`
        x0 = fast_obs_x0(f, y[idx - 3], y[idx - 2], y[idx - 1], up[idx - 1], up[idx - 2])
        x1 = E0 @ x0 + E1 * u
        yi = C @ x1
        y = y.at[idx].set(yi)
        return y

    y = jax.lax.fori_loop(0, data.size, filt_body, y)
    y = y[50:]  # remove initial padding
    return y

## opt

In [ ]:
def cost(
    params: jax.Array,
    data: jax.Array,
    ref_data: jax.Array,
    ref_datap: jax.Array,
) -> jax.Array:
    nn = build_conv_net(params)

    data_filt = nn_linear_filt(nn, data)
    cost = jnp.mean(jnp.square(data_filt - ref_data)) * 1e3

    # data_filtp = nn_linear_filt(nn, data, nu=1)
    # cost += jnp.mean(jnp.square(data_filtp - ref_datap)) * 1e-1
    return cost

cost_vg = jax.jit(jax.value_and_grad(cost))

In [ ]:
res = sci_opt.minimize(
    fun=functools.partial(cost_vg, data=data, ref_data=ref_data, ref_datap=ref_datap),
    x0=params0,
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 20,
    }
)

In [ ]:
if False:
    res = sci_opt.minimize(
        fun=functools.partial(cost_vg, data=data, ref_data=ref_data, ref_datap=ref_datap),
        x0=res.x,
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 200,
        }
    )

In [ ]:
opt_conv = build_conv_net(res.x)

## analysis

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class StupidNN:
    def __call__(self, _):
        # return -2.0 * np.pi
        return -4.0
        # return -10.0

stupid_nn = StupidNN()

In [ ]:
plot_range = [0, 150 * 200]
# plot_range = [150 * 200, 250 * 200]
# plot_range = [1000 * 200, 1100 * 200]

plot_data = acc_ref[slice(*plot_range), 0]
# plot_data = test_acc_ref[slice(*plot_range), 0]
plot_ref_data = smooth_data(plot_data)

opt_filt = nn_linear_filt(opt_conv, data=plot_data)
ref_filt = nn_linear_filt(stupid_nn, data=plot_data)

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(plot_data, label="plot_data", alpha=0.2)
ax.plot(plot_ref_data, label="plot_ref_data", alpha=0.4)
ax.plot(opt_filt, label="opt_filt")
ax.plot(ref_filt, label="ref_filt")

# ax.set_ylim(-1, 1)
ax.legend()
ax.grid()

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def nn_cutoffs(
    nn: ConvNet,
    data: jax.Array,
    nu: int=0,
) -> jax.Array:
    y = jnp.zeros(data.size + 50)
    cutoffs = jnp.zeros(data.size)
    up = jnp.concatenate([jnp.ones(50) * data[0], data])

    def filt_body(i: int, state: tuple[jax.Array, jax.Array]) -> jax.Array:
        y, cutoffs = state
        u_hist = jax.lax.dynamic_slice(up, [i], [50])
        y_hist = jax.lax.dynamic_slice(y, [i], [50])

        f = nn(u_hist - y_hist)
        cutoffs = cutoffs.at[i].set(f)
        E0, E1, C = fast_trip_E0_E1_C(f, nu)

        # technically, the initial y are out of bouds, but zero init is good)
        idx = i + 50
        u = up[idx]  # not included in `u_hist`
        x0 = fast_obs_x0(f, y[idx - 3], y[idx - 2], y[idx - 1], up[idx - 1], up[idx - 2])
        x1 = E0 @ x0 + E1 * u
        yi = C @ x1
        y = y.at[i].set(yi)
        return y, cutoffs

    _, cutoffs = jax.lax.fori_loop(0, y.size, filt_body, (y, cutoffs))
    return cutoffs

In [ ]:
opt_cutoffs = nn_cutoffs(opt_conv, plot_data)
fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.plot(opt_cutoffs)
ax.grid()